In [1]:
!pip install selenium
!pip install openpyxl
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

driver = webdriver.Chrome()
driver.maximize_window()

#driver.get("https://landing.iitb.ac.in/?url=https://asc.iitb.ac.in/")
driver.get("https://asc.iitb.ac.in/acadmenu/")

# login manually in the opened browser
input(" After logging in and reaching home page, press ENTER...")

 After logging in and reaching home page, press ENTER... 


''

In [ ]:
# Import libraries
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time



wait = WebDriverWait(driver, 20)

# STEP 1: Switch to LEFT FRAME (menu)
wait.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "leftPage")))


# STEP 2: Navigate to running courses page
wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "Academic"))).click()
time.sleep(1)

wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "All About Courses"))).click()
time.sleep(1)

wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "Running Courses"))).click()

time.sleep(2)


# STEP 3: Switch to RIGHT FRAME 
driver.switch_to.default_content()  # go back to main
wait.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "rightPage")))


# Function to scrape one semester
def scrape_semester(semester_text, max_courses=5):
    data = []

    #Select Semester and month
    dropdown = Select(driver.find_element(By.XPATH, "//select[contains(@name,'semester')]"))
    dropdown.select_by_visible_text(semester_text)
    driver.find_element(By.XPATH, "//input[@value='GO' or @value='Go']").click()
    time.sleep(2)
    driver.switch_to.default_content()
    wait.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "rightPage")))
    
    #Click on AE
    # wait.until(EC.element_to_be_clickable(
    #     (By.XPATH, "//table//a")
    # )).click()
    # time.sleep(2)

    # Get all department links count first
    dept_links = driver.find_elements(By.XPATH, "//table//a")
    num_departments = len(dept_links)
    
    print(f"Found {num_departments} departments")
    
    for d in range(num_departments):
        # Re-fetch elements every time (prevents stale element error)
        dept_links = driver.find_elements(By.XPATH, "//table//a")
        
        dept_name = dept_links[d].text.strip()
        print(f"Processing Department: {dept_name}")
    
        driver.execute_script("arguments[0].click();", dept_links[d])
        time.sleep(2)
    
       
        
    
        for idx in range(max_courses):
            rows = driver.find_elements(By.XPATH, "//table/tbody/tr")
            valid_rows = []
            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) < 9:
                    continue
                try:
                    # cols[2].find_element(By.TAG_NAME, "a")
                    # valid_rows.append(row)
                    links = cols[2].find_elements(By.TAG_NAME, "a")
                    if len(links) > 0:
                        valid_rows.append(row)    
                except NoSuchElementException:
                    continue
    
            if idx >= len(valid_rows):
                print(f"  ⚠  Only {len(valid_rows)} valid rows found.")
                break
            
            cols = valid_rows[idx].find_elements(By.TAG_NAME, "td")
            course_code = cols[2].text.strip()
            course_name = cols[3].text.strip()
            course_type = cols[4].text.strip()
            category    = cols[5].text.strip()
            instructor  = cols[6].text.strip()
            slot        = cols[8].text.strip()
            division    = cols[9].text.strip() if len(cols) > 9 else ""
    
            link = cols[2].find_element(By.TAG_NAME, "a")
            driver.execute_script("arguments[0].click();", link)
            time.sleep(2)
    
            try:
                credits = driver.find_element(By.XPATH,
                    "//td[normalize-space(text())='Total Credits']/following-sibling::td[1]"
                ).text.strip()
            except:
                credits = ""
            try:
                description = driver.find_element(By.XPATH,
                    "//td[normalize-space(text())='Description']/following-sibling::td[1]"
                ).text.strip()
            except:
                description = ""
    
            driver.back()
            time.sleep(1)
    
            driver.switch_to.default_content()
            wait.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "rightPage")))
    
            data.append({
                "Course Code": course_code, "Course Name": course_name,
                "Course Type": course_type, "Course Category": category,
                "Instructor": instructor, "Slot": slot, "Division": division,
                "Credits": credits, "Description": description,
            })
            print(f"  [{idx+1}/{max_courses}] {course_code} – {course_name} | Credits: {credits}| {course_type} | {category} | {instructor} | {slot} | {division} | {description}")
             # Go back to department list
        driver.back()
        time.sleep(1)
    
        driver.switch_to.default_content()
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "rightPage")))
            
    #Go back to Running courses screen
    driver.back()
    driver.switch_to.default_content()
    wait.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "rightPage")))
    return pd.DataFrame(data)



    
df_sem2 = scrape_semester("2 - Spring",5)
df_sem1 = scrape_semester("1 - Autumn",5)

# Save to Excel (2 sheets)
file_name = "25b2244_q_4.xlsx"

with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    df_sem1.to_excel(writer, sheet_name="2025 1", index=False)
    df_sem2.to_excel(writer, sheet_name="2025 2", index=False)

print("Data saved successfully!")

# Close browser
driver.quit()

Found 32 departments
Processing Department: AE,AES
  [1/5] AE 152 – Introduction to Aerospace Engg. | Credits: 6.0| Theory | Interdisciplinary STEM Course. | I - Vasudeva Raghavendra Kowsik Bodi | 10
Fri-10B-14:00:00-15:25:00-Class Room : LC 202
Tue-10A-14:00:00-15:25:00-Class Room : LC 202 |  | Nomenclature of aircraft components. Standard atmosphere. Basic Aerodynamics : Streamlines, steady fluid motion, incompressible flow, Bernoulli"s equation,Mach number, Pressure and airspeed measurement, Boundary Layer,Reynolds number, Laminar and Turbulent flow. Airfoils and wings: pressure coefficient and lift calculation, Critical Mach number, Wave drag, Finite wings, Induced drag, Swept wings. Aircraft performance: steady level flight, Altitude effects, Absolute ceiling, steady climbing flight, Energy methods, Range and Endurance, Sustained level turn, pull-up, V-n diagram, Take off and landing. Reentry vehicles: Ballistic and Glide Reentry, Blunt body concept.
  [2/5] AE 153 – Introduction 